# Lab 8 — Bug hunt, Grad-CAM, and a robot that sorts parts

**Deep Learning for Computer Vision — Day 2** · ECAM, 5MAU · 2026–2027

Most deep-learning bugs **do not crash**: the code runs, the loss decreases a bit, and the model is just bad. This lab trains you to spot them — the exam contains this kind of exercise.

**Part 1 — Bug hunt (~30 min).** Each cell contains a short piece of code with **one bug** (sometimes two). Find it, fix it in place, and run the cell until the `assert` passes. Explain the bug in one sentence in the markdown cell below it.

**Part 2 — Grad-CAM and inference (~15 min).** Where does the inspection model of Lab 7 look? How fast is it?

**Part 4 — Sorting parts with the robot arm (~30 min, at the robot cell).** The CNN of Lab 7 meets the camera and the Galatae arm of Lab 4: find the parts, classify them, pick and place them in the right bin.

*If time is short:* bugs 7–9 and Part 3 can be finished at home; Part 4 can be continued during the project.

> *References:* A. Karpathy, "A recipe for training neural networks", 2019, https://karpathy.github.io/2019/04/25/recipe/ · R. Selvaraju *et al.*, "Grad-CAM: Visual explanations from deep networks via gradient-based localization", ICCV 2017 · PyTorch documentation, https://pytorch.org/docs/ · KenN7, *Deep learning for computer vision*, ECAM, lab 4 (*inference & validation*). · Galatae, *galatae-api* (Python API of the Gal-e arm), https://github.com/galatae-engineering/galatae-api

In [ ]:
# Setup check: are all the packages of requirements.txt installed in this kernel?
import importlib
import importlib.util

importlib.invalidate_caches()
REQUIRED = {"torch": "torch", "torchvision": "torchvision", "numpy": "numpy", "matplotlib": "matplotlib",
            "cv2": "opencv-contrib-python", "ipywidgets": "ipywidgets",
            "serial": "pyserial"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
assert not missing, (f"Missing packages: {missing}. Run `%pip install -r ../requirements.txt` "
                     "in a cell, then restart the kernel (Kernel > Restart) and run the notebook again.")
print("Setup OK: all required packages are installed.")

In [ ]:
from pathlib import Path
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms as T

DATA = Path("data") if Path("data").exists() else Path("../data")
PARTS = DATA / "parts"
torch.manual_seed(0)

# A tiny, linearly separable 2-class dataset used by several bugs
X = torch.randn(512, 2)
Y = (X[:, 0] + X[:, 1] > 0).long()

## Bug 1 — The loss explodes

In [ ]:
torch.manual_seed(0)
model = nn.Linear(2, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.5)
for step in range(100):
    logits = model(X)
    loss = F.cross_entropy(logits, Y)
    loss.backward()
    opt.step()
print("final loss", loss.item())
assert loss.item() < 0.2

*Your explanation:* …

## Bug 2 — The model is (almost) right but learns very slowly

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2), nn.Softmax(dim=1))
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for step in range(200):
    loss = F.cross_entropy(model(X), Y)
    opt.zero_grad(); loss.backward(); opt.step()
print("final loss", loss.item())
out = model(X[:5])
assert not torch.allclose(out.sum(1), torch.ones(5)), "the model must output logits, not probabilities"
assert loss.item() < 0.1

*Your explanation:* …

## Bug 3 — Two evaluations of the same data give two different results

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, 2))


def accuracy(model, x, y):
    model.train()
    logits = model(x)
    return (logits.argmax(1) == y).float().mean().item()


a1, a2 = accuracy(model, X, Y), accuracy(model, X, Y)
print(a1, a2)
assert a1 == a2

*Your explanation:* …

## Bug 4 — The validation accuracy changes at every evaluation

In [ ]:
MEAN, STD = 0.30, 0.25
train_tfm = T.Compose([T.Grayscale(), T.RandomRotation(180), T.ColorJitter(brightness=0.4),
                       T.ToTensor(), T.Normalize([MEAN], [STD])])
val_tfm = train_tfm
val_set = datasets.ImageFolder(PARTS / "val", val_tfm)
assert torch.equal(val_set[0][0], val_set[0][0]), "loading the same validation image twice must give the same tensor"

*Your explanation:* …

## Bug 5 — Inference on an image from the camera

The inspection model of Lab 7 is loaded (or quickly retrained if `parts_cnn.pt` is not found). A new image arrives from the camera as an OpenCV array. The model predicts nonsense…

In [ ]:
def block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(), nn.MaxPool2d(2))


class GapCNN(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        self.features = nn.Sequential(block(1, 16), block(16, 32), block(32, 64))
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


classes = datasets.ImageFolder(PARTS / "val").classes
model = GapCNN()
if Path("parts_cnn.pt").exists():
    model.load_state_dict(torch.load("parts_cnn.pt", map_location="cpu"))
    print("loaded parts_cnn.pt from Lab 7")
else:  # quick retraining (~1 min)
    print("parts_cnn.pt not found: training a model (about 1 minute)...")
    loader = torch.utils.data.DataLoader(datasets.ImageFolder(PARTS / "train", train_tfm), batch_size=32, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 3e-3, epochs=12, steps_per_epoch=len(loader))
    for epoch in range(12):
        model.train()
        for x, y in loader:
            loss = F.cross_entropy(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
model.eval();

In [ ]:
def predict(model, img_bgr):
    """img_bgr: uint8 image (H, W, 3) from cv2.imread / camera. Returns the predicted class name."""
    x = torch.from_numpy(img_bgr).float()
    x = x.unsqueeze(0)
    with torch.no_grad():
        return classes[model(x).argmax(1).item()]


files = sorted((PARTS / "test").glob("*/*.png"))[::16]
hits = [predict(model, cv2.imread(str(f))) == f.parent.name for f in files]
print(f"{sum(hits)}/{len(hits)} correct")
assert np.mean(hits) > 0.85

*Your explanation:* …

## Bug 6 — The network does not accept 64×64 images

In [ ]:
class SmallNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 8, 3), nn.ReLU(), nn.MaxPool2d(2),
                                      nn.Conv2d(8, 16, 3), nn.ReLU(), nn.MaxPool2d(2))
        self.fc = nn.Linear(16 * 16 * 16, 4)

    def forward(self, x):
        return self.fc(self.features(x).flatten(1))


assert SmallNet()(torch.zeros(2, 1, 64, 64)).shape == (2, 4)

*Your explanation:* …

## Bug 7 — The accuracy is wrong

In [ ]:
logits = torch.tensor([[2.0, 0.1, 0.3], [0.2, 0.1, 3.0], [0.5, 1.5, 0.2], [1.0, 0.9, 0.8]])
labels = torch.tensor([0, 2, 1, 2])
acc = (logits.argmax(dim=0) == labels).float().mean().item()
print("accuracy", acc)
assert acc == 0.75

*Your explanation:* …

## Bug 8 — Out of memory after a few hundred iterations

In [ ]:
torch.manual_seed(0)
net = nn.Linear(2, 2)
opt = torch.optim.SGD(net.parameters(), lr=0.1)
running = 0.0
for step in range(50):
    loss = F.cross_entropy(net(X), Y)
    opt.zero_grad(); loss.backward(); opt.step()
    running += loss
assert isinstance(running, float), "running must be a Python float"

*Your explanation:* …

## Bug 9 — Preprocessing a camera image

In [ ]:
img = np.full((4, 4), 100, np.uint8)
background = np.full((4, 4), 120, np.uint8)
diff = img - background            # "how much darker than the background?"
print(diff[0, 0])
assert diff[0, 0] == -20

*Your explanation:* …

## Part 2 — Grad-CAM: where does the model look?

**Grad-CAM** weights each feature map $A^k$ of the last convolutional layer by the average gradient of the class score $y^c$ with respect to it:
$$ \alpha_k^c = \frac{1}{HW}\sum_{i,j} \frac{\partial y^c}{\partial A^k_{ij}}, \qquad L^c = \mathrm{ReLU}\Big(\sum_k \alpha_k^c A^k\Big). $$
The map $L^c$ (8×8 here) is upsampled to the image size and overlaid: it shows which regions increased the score of class $c$.

### Exercise 10.1
Complete `grad_cam`. The forward hook stores the activations $A$ and a hook on the tensor stores its gradient.

In [ ]:
def grad_cam(model, x, target_layer, class_idx=None):
    store = {}

    def fwd_hook(module, inp, out):
        store["A"] = out
        out.register_hook(lambda g: store.__setitem__("dA", g))

    h = target_layer.register_forward_hook(fwd_hook)
    model.eval()
    logits = model(x)
    h.remove()
    c = logits.argmax(1).item() if class_idx is None else class_idx
    model.zero_grad()
    logits[0, c].backward()
    A, dA = store["A"][0].detach(), store["dA"][0]  # (K, h, w)
    # TODO
    cam = ...   (h, w) tensor, >= 0, normalised to [0, 1]
    return cam, c


tfm = T.Compose([T.Grayscale(), T.ToTensor(), T.Normalize([MEAN], [STD])])
test_set = datasets.ImageFolder(PARTS / "test", tfm)
idx = [i for c in ["missing_hole", "scratch", "stain"]
       for i in [j for j, (_, y) in enumerate(test_set.samples) if classes[y] == c][:3]]
fig, ax = plt.subplots(2, len(idx), figsize=(2 * len(idx), 4.4))
for k, i in enumerate(idx):
    x, y = test_set[i]
    cam, c = grad_cam(model, x.unsqueeze(0), model.features[2])
    cam = cv2.resize(cam.numpy(), (64, 64), interpolation=cv2.INTER_LINEAR)
    ax[0, k].imshow(x[0] * STD + MEAN, cmap="gray"); ax[0, k].set_title(f"true {classes[y]}", fontsize=8)
    ax[1, k].imshow(x[0] * STD + MEAN, cmap="gray"); ax[1, k].imshow(cam, cmap="jet", alpha=0.45)
    ax[1, k].set_title(f"pred {classes[c]}", fontsize=8)
for a in ax.ravel():
    a.axis("off")

**Question.** Does the model look at the defect? Why is this check important before deploying a model (think of a model that would detect "scratch" from the background or the lighting)?

## Part 3 — Inference speed

On the line, a decision is needed for every part. Measure the latency of the model on the CPU for one image and the throughput for batches.

In [ ]:
model.eval()
with torch.no_grad():
    x1 = torch.zeros(1, 1, 64, 64)
    for _ in range(5):
        model(x1)  # warm-up
    for bs in [1, 8, 64]:
        xb = torch.zeros(bs, 1, 64, 64)
        t0 = time.perf_counter()
        for _ in range(20):
            model(xb)
        dt = (time.perf_counter() - t0) / 20
        print(f"batch {bs:3d}: {1000 * dt:6.1f} ms per batch, {1000 * dt / bs:5.2f} ms per image")

**Question.** Why is the time per image smaller with larger batches? When can you *not* use batches on a production line?

## Part 4 — Sorting parts with the robot arm

Back to the robot cell of Lab 4: same camera above the table, same 4 ArUco markers, same **Galatae Gal-e** arm. Printed flanges (60 × 60 mm tiles) lie on the table. The cell must **pick each part and drop it in the right bin**: `ok` parts in the good bin, defective parts in the reject bin, and parts on which the model hesitates in a bin checked by an operator.

$$\text{image} \xrightarrow{\ \text{undistort} + H\ } \text{top view (mm)} \xrightarrow{\ \text{find tiles}\ } \text{crops } 64 \times 64 \xrightarrow{\ \text{CNN}\ } \text{class} \xrightarrow{\ \text{decision}\ } \text{pick \& place}$$

Day 1 tells us **where** the parts are, the CNN tells us **what** they are. We work in simulation on `data/robot/cell_parts_*.png` (same simulated camera and robot as in Lab 4); section 14 runs the same code with the real camera and the real arm.

In [ ]:
import copy
import json

ROBOT_DATA = DATA / "robot"
cam = json.loads((ROBOT_DATA / "cell_camera.json").read_text())  # calibration of Lab 4
K_cell, dist_cell = np.array(cam["K"]), np.array(cam["dist"])
MARKERS_MM = {0: (0.0, 0.0), 1: (400.0, 0.0), 2: (400.0, 280.0), 3: (0.0, 280.0)}  # marker centres, table frame
detector = cv2.aruco.ArucoDetector(cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50))
PPM = 2.0                              # pixels per mm of the top view
AREA = (-30.0, -30.0, 430.0, 310.0)    # part of the table shown in the top view: X0, Y0, X1, Y1 (mm)


def top_view(img, K, dist, markers_mm=MARKERS_MM, ppm=PPM, area=AREA):
    """Undistort the image and warp the table plane to a top view where 1 mm = ppm pixels (Day 1, Lab 3-4)."""
    corners, ids, _ = detector.detectMarkers(img)
    assert ids is not None and set(markers_mm) <= set(ids.ravel()), "all the markers must be visible"
    centres = {int(i): c.reshape(4, 2).mean(axis=0) for c, i in zip(corners, ids.ravel())}
    uv = np.array([centres[i] for i in markers_mm], np.float32).reshape(-1, 1, 2)
    uv = cv2.undistortPoints(uv, K, dist, P=K).reshape(-1, 2)
    xy = np.array([[(markers_mm[i][0] - area[0]) * ppm, (markers_mm[i][1] - area[1]) * ppm] for i in markers_mm],
                  np.float32)
    H_top = cv2.getPerspectiveTransform(uv, xy)
    size = (int((area[2] - area[0]) * ppm), int((area[3] - area[1]) * ppm))
    return cv2.warpPerspective(cv2.undistort(img, K, dist), H_top, size)


def top_to_table(p, ppm=PPM, area=AREA):
    """Top-view pixel(s) -> table coordinates (X, Y) in mm."""
    return np.asarray(p) / ppm + np.array(area[:2])


img = cv2.imread(str(ROBOT_DATA / "cell_parts_1.png"))
top = top_view(img, K_cell, dist_cell)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].imshow(img[:, :, ::-1]); ax[0].set_title("camera image"); ax[0].axis("off")
ax[1].imshow(top[:, :, ::-1], extent=[AREA[0], AREA[2], AREA[3], AREA[1]])
ax[1].set(title="top view of the table", xlabel="X [mm]", ylabel="Y [mm]");

### Exercise 11 — Find the parts
The tiles are dark squares on a light table: an Otsu threshold (Day 1) separates them, but the black squares of the ArUco markers are dark too. Complete `find_tiles`: keep the contours whose **rotated bounding box** (`cv2.minAreaRect`, which returns `((cx, cy), (w, h), angle)` in top-view pixels) has both sides within ±15 % of 60 mm. The markers are 40 mm wide.

In [ ]:
TILE_MM = 60.0


def find_tiles(top, tile_mm=TILE_MM, ppm=PPM, tol=0.15):
    """Rotated rectangles ((cx, cy), (w, h), angle) of the tiles, in top-view pixels."""
    gray = cv2.cvtColor(top, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    tiles = []
    for c in contours:
        rect = cv2.minAreaRect(c)
        # TODO
        ...
    return tiles


def crop_tile(top, rect, size=64):
    """Warp a (rotated) tile to a size x size image: the same framing as the training images."""
    src = cv2.boxPoints(rect).astype(np.float32)
    dst = np.array([[0, size], [0, 0], [size, 0], [size, size]], np.float32)
    return cv2.warpPerspective(top, cv2.getPerspectiveTransform(src, dst), (size, size), flags=cv2.INTER_AREA)


tiles = find_tiles(top)
crops = [crop_tile(top, r) for r in tiles]
centres_table = top_to_table([r[0] for r in tiles])
print(f"{len(tiles)} tiles found; centres (table, mm):\n", centres_table.round(1))
gt_cell = json.loads((ROBOT_DATA / "ground_truth.json").read_text())
true_tiles = {s["file"]: s["tiles"] for s in gt_cell["scenes"]}


def match_truth(file, centres):
    """True tile (dict) closest to each detected centre (to check the results)."""
    tt = true_tiles[file]
    return [tt[int(np.argmin([np.hypot(*(c - t["centre_table_mm"])) for t in tt]))] for c in centres]


truth = match_truth("cell_parts_1.png", centres_table)
errors = [np.hypot(*(c - t["centre_table_mm"])) for c, t in zip(centres_table, truth)]
print("position errors [mm]:", np.round(errors, 2))
assert len(tiles) == 5 and max(errors) < 2.0
fig, ax = plt.subplots(1, 5, figsize=(12, 2.6))
for a, c, t in zip(ax, crops, truth):
    a.imshow(c[:, :, ::-1]); a.set_title(f"true: {t['class']}", fontsize=9); a.axis("off")

### Exercise 12.1 — Classify the crops
Complete `classify_crops`: one **batch** with all the crops of the image (Part 3), the **same preprocessing as in training** (Bug 5), and the softmax probabilities as output.

In [ ]:
def classify_crops(model, crops):
    """List of BGR uint8 crops (64 x 64) -> (N, 4) numpy array of class probabilities."""
    model.eval()
    # TODO
    probs = ...
    return probs


probs = classify_crops(model, crops)
assert probs.shape == (5, 4) and np.allclose(probs.sum(1), 1, atol=1e-4)
for t, p in zip(truth, probs):
    print(f"true {t['class']:13s} predicted {classes[p.argmax()]:13s} (p = {p.max():.2f})")

**Question.** The model of Lab 7 reached ~97 % on the test set. How does it do here? The crops look fine to a human: compare them with the test images below. What differs between the images of the dataset and the camera images?

In [ ]:
test_files = sorted((PARTS / "test").glob("*/*.png"))[::8]
for sigma in [0, 0.5, 0.8, 1.2]:
    ims = [cv2.imread(str(f)) for f in test_files]
    if sigma > 0:
        ims = [cv2.GaussianBlur(im, (0, 0), sigma) for im in ims]
    p = classify_crops(model, ims)
    acc = np.mean([classes[k] == f.parent.name for k, f in zip(p.argmax(1), test_files)])
    print(f"test images blurred with sigma = {sigma}: accuracy {acc:.2f}")
fig, ax = plt.subplots(1, 4, figsize=(8, 2.3))
for a, im, title in zip(ax, [cv2.imread(str(test_files[0])), crops[0], cv2.imread(str(test_files[20])), crops[1]],
                        ["test image", "camera crop", "test image", "camera crop"]):
    a.imshow(im[:, :, ::-1]); a.set_title(title, fontsize=9); a.axis("off")

### Exercise 12.2 — Adapt the model to the camera
The fix is the one of Lab 7: make the training data cover the deployment conditions. Add a random Gaussian blur (`T.RandomApply([T.GaussianBlur(5, sigma=(0.5, 1.5))], p=0.7)`) to the augmentations and **fine-tune** a copy of the model for a few epochs (≈ 20 s).

In [ ]:
cell_tfm = T.Compose([
    T.Grayscale(), T.RandomRotation(180), T.RandomAffine(0, translate=(0.08, 0.08)),
    T.ColorJitter(brightness=0.4), T.RandomHorizontalFlip(),
    # TODO
    # add a random Gaussian blur here (T.RandomApply + T.GaussianBlur)
    T.ToTensor(), T.Normalize([MEAN], [STD]),
])
torch.manual_seed(0)
cell_model = copy.deepcopy(model)
loader = torch.utils.data.DataLoader(datasets.ImageFolder(PARTS / "train", cell_tfm), batch_size=32, shuffle=True)
opt = torch.optim.AdamW(cell_model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, 1e-3, epochs=4, steps_per_epoch=len(loader))
t0 = time.perf_counter()
for epoch in range(4):
    cell_model.train()
    for x, y in loader:
        loss = F.cross_entropy(cell_model(x), y)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
cell_model.eval()
test_eval = datasets.ImageFolder(PARTS / "test", tfm)
xt, yt = torch.stack([s[0] for s in test_eval]), torch.tensor(test_eval.targets)
with torch.no_grad():
    test_acc = (cell_model(xt).argmax(1) == yt).float().mean().item()
print(f"fine-tuned in {time.perf_counter() - t0:.0f} s | test accuracy {test_acc:.3f}")

hits = 0
for f in sorted(ROBOT_DATA.glob("cell_parts_*.png")):
    top_f = top_view(cv2.imread(str(f)), K_cell, dist_cell)
    tiles_f = find_tiles(top_f)
    p = classify_crops(cell_model, [crop_tile(top_f, r) for r in tiles_f])
    truth_f = match_truth(f.name, top_to_table([r[0] for r in tiles_f]))
    hits += sum(classes[k] == t["class"] for k, t in zip(p.argmax(1), truth_f))
    print(f.name, [f"{classes[k]} ({q.max():.2f})" for k, q in zip(p.argmax(1), p)])
print(f"cell images: {hits}/15 parts correctly classified")
assert test_acc > 0.9 and hits >= 12

### Exercise 13 — Decide and sort
A wrong decision has a cost: a defective part shipped to a customer is much worse than a good part checked by an operator. Complete `decide`: `"ok"` or `"reject"` when the most probable class has a probability $\geq$ `threshold`, `"check"` (operator) otherwise.

Then complete `pick_and_place` with the **safe sequence** of Lab 4, tool pointing down (`pitch = 180`, `roll = 0`) and the gripper angle as 6th value of the pose: open above the part → descend → close → lift → above the bin → open (drop) → stay above the bin.

In [ ]:
def decide(p, threshold=0.8):
    """Probabilities of one part -> "ok", "reject" or "check"."""
    # TODO
    ...


assert decide(np.array([0.05, 0.9, 0.03, 0.02])) == "ok"
assert decide(np.array([0.85, 0.1, 0.03, 0.02])) == "reject"
assert decide(np.array([0.3, 0.6, 0.05, 0.05])) == "check"

# Robot cell (robot frame, mm): table taught in Lab 4, bins beside the table
taught = json.loads((ROBOT_DATA / "taught_markers.json").read_text())["markers"]
xyz_taught = np.array([taught[str(i)] for i in range(4)])
M_aff = np.hstack([np.array([MARKERS_MM[i] for i in range(4)]), np.ones((4, 1))])
A_tr = np.linalg.lstsq(M_aff, xyz_taught[:, :2], rcond=None)[0].T   # table -> robot affine map (Lab 4, ex. 9.1)
z_table = xyz_taught[:, 2].mean()
BINS = {"ok": (200.0, 285.0), "reject": (285.0, 285.0), "check": (370.0, 285.0)}  # (x, y) robot frame
GRIPPER_OPEN, GRIPPER_CLOSED = 60.0, 10.0  # gripper angle (deg): check the values on your arm!
REACH_MM = (120.0, 550.0)
SAFE_DZ, GRASP_DZ, DROP_DZ = 80.0, 3.0, 60.0  # heights above the table (mm)


def table_to_robot_xy(xy):
    return np.atleast_2d(xy) @ A_tr[:, :2].T + A_tr[:, 2]


def safe_move(robot, pose, z_table):
    x, y, z = pose[:3]
    assert REACH_MM[0] <= np.hypot(x, y) <= REACH_MM[1], f"out of reach: {pose}"
    assert z >= z_table + 2, f"too low: z = {z:.1f} mm (table at {z_table:.1f} mm)"
    robot.go_to_pose([float(v) for v in pose])


class SimulatedRobot:
    """Same interface as galatae-api's Robot: prints the G-code instead of moving an arm."""

    def __init__(self, verbose=True):
        self.pose, self.history, self.verbose = [300.0, 0.0, 200.0, 180.0, 0.0, GRIPPER_OPEN], [], verbose

    def _send(self, gcode):
        if self.verbose:
            print("  ->", gcode)

    def reset(self):
        self._send("$RST=*")

    def home(self):
        self._send("$H")

    def set_joint_speed(self, speed):
        self._send(f"F{speed}")

    def go_to_pose(self, pose):
        self._send("G1" + "".join(f"{w}{round(v, 2)}" for w, v in zip("XYZABC", pose)))
        self.pose = list(pose)
        self.history.append(list(pose))

    def get_tool_pose(self):
        return list(self.pose)

    def go_to_sleep_pos(self):
        self._send("G28")


def pick_and_place(robot, xy_part, xy_bin, z_table):
    """Pick the part at xy_part (robot frame) and drop it above xy_bin."""
    (x, y), (xb, yb) = xy_part, xy_bin
    # TODO
    ...


def sort_cell(robot, img, model, K, dist, markers_mm=MARKERS_MM, threshold=0.8):
    """The whole cell: image -> parts -> decisions -> robot. Returns the list of (xy_robot, decision)."""
    top = top_view(img, K, dist, markers_mm)
    tiles = find_tiles(top)
    if not len(tiles):
        print("no tile found: the robot does not move")
        return top, tiles, []
    probs = classify_crops(model, [crop_tile(top, r) for r in tiles])
    xy_robot = table_to_robot_xy(top_to_table([r[0] for r in tiles]))
    plan = [(xy, decide(p, threshold), p) for xy, p in zip(xy_robot, probs)]
    for xy, d, p in plan:
        print(f"part at ({xy[0]:.0f}, {xy[1]:.0f}) mm: {classes[p.argmax()]} (p = {p.max():.2f}) -> bin '{d}'")
        pick_and_place(robot, xy, BINS[d], z_table)
    return top, tiles, plan


robot = SimulatedRobot(verbose=False)
img = cv2.imread(str(ROBOT_DATA / "cell_parts_2.png"))
top, tiles, plan = sort_cell(robot, img, cell_model, K_cell, dist_cell)
traj = np.array(robot.history)
print(f"{len(traj)} poses sent; first part:")
for pose in traj[:6]:
    print("  G1" + "".join(f"{w}{round(v, 1)}" for w, v in zip("XYZABC", pose)))
assert len(traj) == 6 * len(plan) and traj[:, 2].min() >= z_table + 2

colours = {"ok": "lime", "reject": "red", "check": "orange"}
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(top[:, :, ::-1])
for r, (xy, d, p) in zip(tiles, plan):
    ax[0].add_patch(plt.Polygon(cv2.boxPoints(r), fill=False, ec=colours[d], lw=2.5))
    ax[0].text(r[0][0], r[0][1] - 70, f"{classes[p.argmax()]} {p.max():.2f}", color=colours[d], ha="center",
               fontsize=9, weight="bold")
ax[0].set_title("decisions (green: ok, red: reject, orange: operator)"); ax[0].axis("off")
ax[1].plot(*xyz_taught[[0, 1, 2, 3, 0], :2].T[::-1], "k--", label="table (taught markers)")
ax[1].plot(traj[:, 1], traj[:, 0], "b-", lw=0.8, alpha=0.6, label="tool path")
for xy, d, p in plan:
    ax[1].plot(xy[1], xy[0], "s", color=colours[d], ms=12, mec="k")
for name, (xb, yb) in BINS.items():
    ax[1].plot(yb, xb, "o", color=colours[name], ms=18, mec="k")
    ax[1].text(yb - 20, xb, f"bin {name}", va="center")
ax[1].plot(0, 0, "bs", ms=12, label="robot base")
ax[1].set(xlabel="y robot [mm]", ylabel="x robot [mm]", title="top view (robot frame)", aspect="equal")
ax[1].invert_xaxis(); ax[1].legend(loc="lower left", fontsize=8)
plt.tight_layout()

**Question.** How would you choose the threshold on a real line? What happens to the number of parts sent to the operator when the threshold increases? The tiles have different orientations: what should change in the pose to grasp a square part by two opposite sides?

### 14 — Live: the real camera and the real arm (groups)

Ask the teacher before running the robot. Low speed, a hand near the emergency stop, nobody in the cell.

1. Print `data/robot/parts_sheet_A4.png` at 100 %, glue it on cardboard and cut the 12 tiles (60 mm). Use the markers and the cell camera of Lab 4 (`cell_calibration.yaml`, `MARKERS_LIVE`).
2. Check the gripper angles `GRIPPER_OPEN` / `GRIPPER_CLOSED` with the teacher (6th value of the pose) and teach the markers as in Lab 4 (the `A_tr`, `z_table` above come from the simulated cell).
3. Place a few tiles (not touching each other) and run the last cell; after the teaching and after each sorting, the arm folds away so that it does not hide the table. Then try new conditions: lighting, a tile partly hidden, a crumpled tile… Does the model hesitate (operator bin) or does it make confident mistakes?

In [ ]:
LIVE_CAMERA = False  # True: take the picture with the USB camera of the cell
LIVE_ROBOT = False   # True: send the commands to the Galatae arm (ask the teacher first!)
CAMERA_ID = 1
ROBOT_PORT = None    # e.g. "COM5" if the arm is not found automatically
MARKERS_LIVE = MARKERS_MM  # measured marker centres (mm) of your cell, as in Lab 4


def leave_view(robot, z_table, clear=150.0):
    """Lift the tool vertically, then fold the arm so that it does not hide the table from the camera."""
    pose = list(robot.get_tool_pose()[:5])  # 5 values: the gripper does not move
    pose[2] = max(pose[2], z_table + clear)
    safe_move(robot, pose, z_table)
    robot.go_to_sleep_pos()


if LIVE_ROBOT:
    import urllib.request

    if not Path("robot.py").exists():  # official Python API of the arm (GPL-3.0), one file
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/galatae-engineering/galatae-api/main/robot.py", "robot.py")
    import serial
    from robot import Robot

    arm = Robot()
    if ROBOT_PORT is not None:
        arm.arduino = serial.Serial(port=ROBOT_PORT, baudrate=115200, timeout=0.1)
    arm.reset()
    arm.home()
    arm.set_joint_speed(20)  # deg/s: slow
    taught_live = {}
    for i in MARKERS_LIVE:
        input(f"Move the tool tip onto the centre of marker {i}, then press Enter")
        taught_live[i] = arm.get_tool_pose()[:3]
    xyz_live = np.array([taught_live[i] for i in MARKERS_LIVE], float)
    M_live = np.hstack([np.array([MARKERS_LIVE[i] for i in MARKERS_LIVE]), np.ones((len(MARKERS_LIVE), 1))])
    A_tr = np.linalg.lstsq(M_live, xyz_live[:, :2], rcond=None)[0].T
    z_table = xyz_live[:, 2].mean()
    leave_view(arm, z_table)  # the tool is still touching the last marker
else:
    arm = SimulatedRobot(verbose=False)

In [ ]:
# Picture -> parts -> decisions -> robot (run this cell again for each new set of tiles)
if LIVE_CAMERA:
    fs = cv2.FileStorage("cell_calibration.yaml", cv2.FILE_STORAGE_READ)  # saved in Lab 4, section 10
    K_live, dist_live = fs.getNode("K").mat(), fs.getNode("dist").mat()
    fs.release()
    cap = cv2.VideoCapture(CAMERA_ID, cv2.CAP_DSHOW)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280); cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    for _ in range(10):
        ok, img = cap.read()
    cap.release()
    assert ok, "no image from the camera"
else:
    K_live, dist_live, img = K_cell, dist_cell, cv2.imread(str(ROBOT_DATA / "cell_parts_3.png"))

top, tiles, plan = sort_cell(arm, img, cell_model, K_live, dist_live, MARKERS_LIVE)
if plan:  # the arm has moved: fold it, clear view for the next picture
    leave_view(arm, z_table)
plt.figure(figsize=(8, 5)); plt.imshow(top[:, :, ::-1]); plt.axis("off")
for r, (xy, d, p) in zip(tiles, plan):
    plt.gca().add_patch(plt.Polygon(cv2.boxPoints(r), fill=False, ec=colours[d], lw=2.5))